# Ejercicio: analizar_texto con mocks

In [ ]:
from unittest.mock import patch, Mock
from src.analizador import analizar_texto

## Caso éxito

In [ ]:
def test_exito():
    mock_resp = Mock()
    mock_resp.text = 'hola\nmundo'
    mock_resp.raise_for_status = Mock()
    with patch('src.analizador.requests.get', return_value=mock_resp) as mock_get:
        lineas, chars = analizar_texto('fake')
        assert lineas == 2
        assert chars == 9
        mock_get.assert_called_once()

## Solución: Fallo y reintento exitoso

def test_fallo_reintento_exitoso():
    """Simula fallo en primer intento y éxito en segundo"""
    import requests
    
    mock_resp_exito = Mock()
    mock_resp_exito.text = 'texto\nrecuperado'
    mock_resp_exito.raise_for_status = Mock()
    
    # Primer intento falla, segundo tiene éxito
    with patch('src.analizador.requests.get') as mock_get:
        mock_get.side_effect = [
            requests.RequestException("Error temporal"),
            mock_resp_exito
        ]
        
        lineas, chars = analizar_texto('fake_url')
        
        # Verificar resultado
        assert lineas == 2
        assert chars == 15
        
        # Verificar que se llamó 2 veces
        assert mock_get.call_count == 2

## Solución: Fallo total (3 intentos)

def test_fallo_total():
    """Simula fallo en los 3 intentos"""
    import requests
    
    with patch('src.analizador.requests.get') as mock_get:
        mock_get.side_effect = requests.RequestException("Error persistente")
        
        # Debe lanzar RuntimeError después de 3 intentos
        try:
            analizar_texto('fake_url')
            assert False, "Debería haber lanzado RuntimeError"
        except RuntimeError as e:
            assert "No se pudo acceder a la URL después de 3 intentos" in str(e)
        
        # Verificar que se intentó 3 veces
        assert mock_get.call_count == 3

## Caso adicional: Timeout

def test_timeout():
    """Simula timeout de conexión"""
    import requests
    
    with patch('src.analizador.requests.get') as mock_get:
        mock_get.side_effect = requests.Timeout("Timeout")
        
        try:
            analizar_texto('fake_url')
            assert False, "Debería haber lanzado RuntimeError"
        except RuntimeError as e:
            assert "No se pudo acceder a la URL después de 3 intentos" in str(e)
        
        assert mock_get.call_count == 3

## Caso adicional: Respuesta con error HTTP

def test_error_http():
    """Simula respuesta HTTP con error 404"""
    import requests
    from unittest.mock import Mock
    
    mock_resp_error = Mock()
    mock_resp_error.raise_for_status.side_effect = requests.HTTPError("404 Not Found")
    
    with patch('src.analizador.requests.get', return_value=mock_resp_error) as mock_get:
        try:
            analizar_texto('fake_url')
            assert False, "Debería haber lanzado RuntimeError"
        except RuntimeError as e:
            assert "No se pudo acceder a la URL después de 3 intentos" in str(e)
        
        assert mock_get.call_count == 3

## Ejecutar todas las pruebas

if __name__ == "__main__":
    test_exito()
    test_fallo_reintento_exitoso()
    test_fallo_total()
    test_timeout()
    test_error_http()
    print("✅ Todas las pruebas de mocks pasaron correctamente")